# 🎯 [AMM02_M08] 7일간 고강도 전면전 모의 탄약/유류 소모량 극한 스트레스 시뮬레이션
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/AMM02/amm02_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 탄약·특수물자 (Level 3) |
| **미션 ID** | AMM02_M08 |
| **미션 주제** | 7일간 고강도 전면전 모의 탄약/유류 소모량 극한 스트레스 시뮬레이션 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
탄약 저장고는 공간이 한정되어 있고 품목마다 함께 둘 수 없는 제약이 있어, 앞으로 얼마나 소모되고 얼마나 자주 반출될지를 알아야 입고 위치와 재보급 순서를 합리적으로 정할 수 있다. 그러나 작전 단계별 고정 기준 소모율(CSR)만으로 계획하면 교전 강도가 낮은 날은 과잉 재고가 쌓이고 높은 날은 긴급 재보급이 반복되어 저장 공간과 작업 동선이 낭비된다. 본 프로젝트는 교육용 가상 부대·가상 품목·가상 저장고 자료를 코드로 만들어, 탄약·유류 소모율 데이터 정의와 전처리, CSR 기반 Baseline과 탐색적 분석, 적 저항·화력 계획을 반영한 소모량 예측, 보급 수준(DOS) 연계 재보급 산출, 작전 지도 대시보드와 설명가능성, 저장 배치 최적화와 군수참모 결재선(HITL), 7일 고강도 스트레스 시험, AAR 기반 소모 계수 재보정을 단계별로 구현한다. 최종 목표는 예측 소모량을 근거로 재보급 소요와 저장 위치를 함께 제안해 저장 효율과 작업 동선을 개선하고, 불출·대체 탄종·재배치는 군수참모가 근거를 확인해 결정하도록 지원하는 통합 시뮬레이터를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(군수참모처, 탄약창)에 즉시 도입 가능한 "완성형 국방 AI 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 행정 및 기술 명세서** | 도메인 데이터 명세서(`docs/amm02_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/amm02_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **군수 데이터 정제 모듈** | 10종 정제 룰 및 긴급 불출 보존 ETL 파이프라인(`src/etl/amm02_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **분석 및 통계 검증** | 90일 소모량 Baseline 비교, 가설 검정 및 ABC 파레토 분석(`notebooks/amm02_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 시계열 파생 피처 엔지니어링 모듈(`src/features/amm02_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 재보급 엔진** | GBR+RF 앙상블 및 DOS 재보급 산출 엔진(`src/models/amm02_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | XAI 기여도 분석 및 자연어 보고서 생성기(`src/explainability/amm02_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **저장 최적화 & HITL** | 정수계획 저장 최적화 및 참모 HITL 결재선 엔진(`src/optimization/amm02_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **작전 지도 대시보드** | 전술 격자 지도 연동 인터랙티브 대시보드 웹앱(`src/dashboard/amm02_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 편차 분석, 축소 추정 CSR 재보정 및 3대 정량 효과 리포트(`reports/amm02_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 훈련 데이터 기반 챔피언/챌린저 모델 자동 성능 비교 및 롤오버 파이프라인(`pipeline/amm02_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 7일간 고강도 전면전 모의 탄약/유류 소모량 극한 스트레스 시뮬레이션
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

7일간 전면전 모의 탄약/유류 소모량 스트레스 시뮬레이션을 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **7일간 고강도 전면전(가상) 상황에서 소모량과 보유량을 일일 모의하고 재보급 경보 대시보드의 성능을 극한 조건에서 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 부대 8개의 마지막 7일을 고강도 공격 단계로 바꿔 소모·보유량을 모의 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/dashboard/amm02_app.py (Streamlit/Dash 등)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 의사결정 지원에 최적화된 시각적 위계 및 사용자 경험(UX)

#### 미션 PASS 조건

- 정량 기준: 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상


- 진행 기준: 웹 기반 분석 대시보드 구동 및 실시간 필터링 기능 검증


- 검증 방법: 실제 군수 데이터 연동 화면 테스트 및 UI 동작 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""7일간 고강도 전면전(가상) 상황에서 부대별 탄약·유류 소모량과 보유량을 일일 모의하고, 보고 누락·보고 지연·긴급 수요 급증·부대 수 10배 부하를 주입해 재보급 경보 대시보드의 갱신 시간(2초 이내)과 경보 발령 정확도(95% 이상)를 검증한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 부대·가상 품목 일일 소모 기록 생성기 (실제 탄약 제원·소모율과 무관)
# 가상 부대 8개 x 90일. 소모량 = 가상 CSR 계수 x 병력 규모 x (교전 강도·적 저항·화력 계획 효과) x 잡음
# ---------------------------------------------------------------------------
UNITS = [f"U{i:02d}" for i in range(1, 9)]
ITEMS = ["ammo_A", "ammo_B", "ammo_C", "ammo_D", "fuel_jet", "fuel_diesel"]
AMMO, FUEL = ITEMS[:4], ITEMS[4:]
PHASES = ["prep", "attack", "defense", "stability"]
CSR = {  # 가상 계수: 병력 100명·1일 기준 소모 단위(탄약=상자, 유류=드럼)
    "prep":      [1.0, 0.6, 0.3, 0.2, 1.5, 6.0],
    "attack":    [6.0, 3.5, 2.0, 1.2, 5.0, 12.0],
    "defense":   [4.5, 3.0, 1.6, 0.8, 3.0, 7.0],
    "stability": [0.8, 0.5, 0.3, 0.2, 1.0, 5.0],
}
PHASE_RESIST = {"prep": 1.0, "attack": 3.2, "defense": 3.6, "stability": 0.8}


def make_ops(n_days=90, seed=7, surge=0):
    """가상 부대별 일일 작전 기록과 품목별 실제 소모량(정답)을 만든다. surge>0이면 마지막 surge일을 고강도 공격 단계로 바꾼다(스트레스 시험용)."""
    rng = np.random.default_rng(seed)
    rows = []
    for uid in UNITS:
        size = int(rng.integers(3, 9)) * 100
        lens = rng.integers(8, 20, 4)
        sched = ["prep"] * lens[0] + ["attack"] * lens[1] + ["defense"] * lens[2] + ["attack"] * lens[3]
        sched = (sched + ["stability"] * n_days)[:n_days]
        if surge:
            sched[-surge:] = ["attack"] * surge
        r = 1.0
        for d, ph in enumerate(sched):
            r = float(np.clip(0.7 * r + 0.3 * (4.8 if surge and d >= n_days - surge else PHASE_RESIST[ph]) + rng.normal(0, 0.35), 0, 5))
            inten = float(np.clip(r / 5 * 0.9 + rng.normal(0, 0.06), 0, 1))
            fire = int(rng.poisson(1 + 8 * inten))
            move = float(rng.gamma(2, {"prep": 4, "attack": 12, "defense": 2, "stability": 5}[ph]))
            weather = str(rng.choice(["clear", "rain", "snow"], p=[0.7, 0.2, 0.1]))
            emerg = bool(rng.random() < 0.02)
            rec = {"unit": uid, "day": d, "phase": ph, "unit_size": size, "intensity": round(inten, 3),
                   "resistance": round(r, 2), "fire_missions": fire, "move_km": round(move, 1),
                   "weather": weather, "emergency": emerg}
            base = size / 100
            for i, it in enumerate(ITEMS):
                c = CSR[ph][i] * base
                if it in AMMO:
                    c *= (0.5 + 0.6 * inten) * (1 + 0.04 * fire * (1.0 if it in ("ammo_C", "ammo_D") else 0.4))
                elif it == "fuel_jet":
                    c *= (0.6 + 0.5 * inten) + 0.03 * fire
                else:
                    c = c * (0.8 + 0.2 * (weather != "clear")) + 0.05 * move * base
                c *= rng.lognormal(0, 0.12) * (2.5 if emerg else 1.0)
                rec[it] = round(float(c), 1)
            rows.append(rec)
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------

# ---- 소모량 예측용 파생 피처: 전일까지의 실적 + 당일 작전·화력 계획(사전에 알 수 있는 값만) ----
FEATS = ["lag1", "lag2", "roll3", "roll7", "csr_base", "size100", "fire_plan", "fire_x_engaged", "resist_lag1",
         "resist_trend", "inten_lag1", "move_plan", "days_in_phase", "engaged", "ph_attack", "ph_defense", "ph_prep",
         "item_idx", "is_fuel"]
DERIVED = ["lag1", "lag2", "roll3", "roll7", "csr_base", "fire_x_engaged", "resist_lag1", "resist_trend",
           "inten_lag1", "days_in_phase", "engaged", "ph_attack", "ph_defense", "ph_prep"]


def build_long(ops):
    """부대x일자 기록을 (부대, 일자, 품목) 행으로 펼치고 파생 피처를 만든다.
    목표값 y = log(실제 소모량 / CSR 기준량): CSR 기준량에 곱할 '보정 계수'를 예측한다. lag·roll 피처도 같은 비율(실적/기준)이다."""
    df = ops.sort_values(["unit", "day"]).reset_index(drop=True)
    g = df.groupby("unit")
    df["resist_lag1"] = g["resistance"].shift(1)
    df["resist_trend"] = df["resist_lag1"] - g["resistance"].shift(3)
    df["inten_lag1"] = g["intensity"].shift(1)
    df["days_in_phase"] = df.groupby((df["phase"] != g["phase"].shift()).cumsum()).cumcount()
    out = []
    for i, it in enumerate(ITEMS):
        csr = np.array([CSR[p][i] for p in df["phase"]])
        ratio = df[it] / (csr * df["unit_size"] / 100)
        prev = ratio.groupby(df["unit"]).shift(1)
        d = pd.DataFrame({"unit": df["unit"], "day": df["day"], "phase": df["phase"], "item": it, "item_idx": i,
                          "is_fuel": int(it in FUEL), "lag1": prev, "lag2": ratio.groupby(df["unit"]).shift(2),
                          "roll3": prev.groupby(df["unit"]).transform(lambda s: s.rolling(3).mean()),
                          "roll7": prev.groupby(df["unit"]).transform(lambda s: s.rolling(7, min_periods=3).mean()),
                          "csr_base": csr, "size100": df["unit_size"] / 100,
                          "fire_plan": df["fire_missions"], "fire_x_engaged": df["fire_missions"] * df["phase"].isin(["attack", "defense"]),
                          "resist_lag1": df["resist_lag1"], "resist_trend": df["resist_trend"], "inten_lag1": df["inten_lag1"],
                          "move_plan": df["move_km"], "days_in_phase": df["days_in_phase"],
                          "engaged": df["phase"].isin(["attack", "defense"]).astype(int),
                          "ph_attack": (df["phase"] == "attack").astype(int), "ph_defense": (df["phase"] == "defense").astype(int),
                          "ph_prep": (df["phase"] == "prep").astype(int), "emergency": df["emergency"],
                          "y": np.log(ratio), "qty": df[it]})
        out.append(d)
    return pd.concat(out, ignore_index=True).dropna(subset=["lag2", "roll3", "resist_trend"]).reset_index(drop=True)


def to_qty(d, y_pred):
    """예측 보정 계수(log)를 소모량으로 되돌린다: 기준량 x exp(예측)."""
    return np.exp(y_pred) * d["csr_base"].values * d["size100"].values


def mape(y, p):
    return float(np.mean(np.abs(np.asarray(p) - np.asarray(y)) / np.asarray(y)))

# ---- 소모량 예측 모델(앙상블) 학습·예측 ----
def fit_models(tr, seed=0):
    """GBR + 랜덤포레스트 두 모델을 학습한다(목표값은 CSR 기준량 대비 log 보정 계수)."""
    g = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, subsample=0.8, random_state=seed).fit(tr[FEATS], tr["y"])
    r = RandomForestRegressor(n_estimators=120, min_samples_leaf=3, random_state=seed, n_jobs=1).fit(tr[FEATS], tr["y"])
    return g, r


def predict_qty(models, d, w=0.5):
    """두 모델의 log 예측을 가중 평균해 소모량으로 되돌린다."""
    g, r = models
    return to_qty(d, w * g.predict(d[FEATS]) + (1 - w) * r.predict(d[FEATS]))

SURGE = 7            # 고강도 모의 일수
DOS_ALERT = 2.0      # [교육용가상자료] 경보 기준 보급 수준(일)

data = build_long(make_ops(90, seed=7, surge=SURGE))
start = 90 - SURGE
models = fit_models(data[data["day"] < start])
test = data[data["day"] >= start].copy()


def inject(d, miss=0.0, delay=0.0, spike=0.0, seed=0):
    """오류 주입: 보고 누락(전일 실적 결측 -> CSR 기준 1.0 대입), 보고 지연(전일 대신 2일 전 실적), 긴급 수요 급증(긴급 불출 요청 접수, 실제 소모 x2.5)."""
    rng = np.random.default_rng(seed); d = d.copy()
    m = rng.random(len(d)) < miss
    d.loc[m, ["lag1", "lag2", "roll3", "roll7"]] = 1.0
    l = (rng.random(len(d)) < delay) & ~m
    d.loc[l, "lag1"] = d.loc[l, "lag2"]
    s = rng.random(len(d)) < spike
    d.loc[s, "qty"] *= 2.5
    d.loc[s, "emergency"] = True
    return d


def dashboard(d, stock0, margin=1.0):
    """대시보드 1회 갱신: 예측 -> 보유량 경과 -> DOS·경보 -> 부대별 집계 표. 반환: 경보표, 집계표."""
    # 긴급 불출 요청이 접수된 건은 요청 배수(가상 2.5배)를 예측에 반영한다(요청은 사전에 알 수 있는 정보)
    d = d.assign(pred=predict_qty(models, d) * np.where(d["emergency"], 2.5, 1.0)).sort_values(["unit", "item", "day"])
    resup = d["csr_base"] * d["size100"] * 1.0          # 계획 재보급량(작전 단계 CSR 기준, 가상)
    used = d.groupby(["unit", "item"])["qty"].cumsum() - d["qty"]
    got = resup.groupby([d["unit"], d["item"]]).cumsum() - resup
    d["on_hand"] = d.set_index(["unit", "item"]).index.map(stock0).values + got.values - used.values
    d["dos_pred"] = d["on_hand"].clip(lower=0) / d["pred"]
    d["dos_true"] = d["on_hand"].clip(lower=0) / d["qty"]
    d["alert"], d["truth"] = d["dos_pred"] < DOS_ALERT * margin, d["dos_true"] < DOS_ALERT
    agg = d.groupby(["day", "unit"]).agg(alerts=("alert", "sum"), ammo=("pred", "sum"))
    return d, agg


stock0 = (test[test["day"] == start].set_index(["unit", "item"])["csr_base"] * test[test["day"] == start].set_index(["unit", "item"])["size100"] * 3.0).to_dict()
cases = [("정상 보고", {}), ("보고 누락 10%", {"miss": 0.1}), ("보고 지연 20%", {"delay": 0.2}),
         ("긴급 급증 5%", {"spike": 0.05}), ("복합(누락10+지연20+급증5)", {"miss": 0.1, "delay": 0.2, "spike": 0.05})]
rows = []
for name, kw in cases:
    d = inject(test, seed=1, **kw)
    t0 = time.perf_counter(); out, agg = dashboard(d, stock0); dt = time.perf_counter() - t0
    tp = int((out["alert"] & out["truth"]).sum()); fn = int((~out["alert"] & out["truth"]).sum())
    rows.append({"시나리오": name, "갱신(s)": dt, "경보": int(out["alert"].sum()), "실제 부족": int(out["truth"].sum()),
                 "정확도": (out["alert"] == out["truth"]).mean(), "부족 탐지율": tp / max(tp + fn, 1)})
res = pd.DataFrame(rows)
print(f"[7일 고강도 모의] 부대 8 x 품목 6 x {SURGE}일 = {len(test)}건, 시작 보유량 3일분(가상)")
print(res.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
acc = res["정확도"].min()
print(f"경보 정확도 최저 {acc:.1%} (목표 95% 이상: {'달성' if acc >= 0.95 else '미달'})")

# 부하 시험: 가상 부대 10배(80개) 데이터로 대시보드 갱신 시간 측정
big = pd.concat([test.assign(unit=test["unit"] + f"_{k}") for k in range(10)], ignore_index=True)
st_big = {(u + f"_{k}", it): v for (u, it), v in stock0.items() for k in range(10)}
t0 = time.perf_counter(); out_b, _ = dashboard(big, st_big); dt_b = time.perf_counter() - t0
mx = max(res["갱신(s)"].max(), dt_b)
print(f"부하 시험(부대 80개, {len(big)}건) 갱신 {dt_b:.2f}s, 최대 갱신 {mx:.2f}s (목표 2초 이내: {'달성' if mx <= 2 else '미달'})")

base, _ = dashboard(test, stock0)
first = base[base["truth"]].groupby("item")["day"].min().sub(start - 1).astype(int)
print("품목별 첫 부족 발생(모의 n일차):", ", ".join(f"{k} {v}일차" for k, v in first.items()) or "없음")
ammo_tot = base[base["is_fuel"] == 0].groupby("day")[["qty", "pred"]].sum()
print(f"7일 탄약 총소모(가상 상자): 실제 {ammo_tot['qty'].sum():.0f}, 예측 {ammo_tot['pred'].sum():.0f}, 오차 {ammo_tot['pred'].sum() / ammo_tot['qty'].sum() - 1:+.1%}")
tr_off = []
for mg in (1.0, 1.1, 1.2):
    o, _ = dashboard(test, stock0, mg)
    tr_off.append(f"x{mg}: 정확도 {(o['alert'] == o['truth']).mean():.1%}/탐지율 {(o['alert'] & o['truth']).sum() / o['truth'].sum():.0%}")
print("경보 안전계수 상충관계(정상 보고):", ", ".join(tr_off), "-> 부족 놓침을 줄이려면 안전계수를 올리고 정확도 저하를 감수한다")
print("경보는 재보급 검토 요청이며 불출·우선순위 결정은 군수참모 결재를 거친다.")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ammo_tot.plot(ax=ax[0], marker="o", title="7-day surge: total ammo (actual vs forecast)")
piv = base.groupby(["unit", "day"])["alert"].sum().unstack()
im = ax[1].imshow(piv.values, cmap="Reds", aspect="auto"); ax[1].set_yticks(range(len(piv))); ax[1].set_yticklabels(piv.index)
ax[1].set_xticks(range(piv.shape[1])); ax[1].set_xticklabels(range(1, piv.shape[1] + 1)); ax[1].set_xlabel("surge day")
ax[1].set_title("Alerts per unit/day (items with DOS<2)"); plt.colorbar(im, ax=ax[1])
ax[2].bar(range(len(res)), res["정확도"]); ax[2].axhline(0.95, c="r", ls="--"); ax[2].set_ylim(0.8, 1.0)
ax[2].set_xticks(range(len(res))); ax[2].set_xticklabels(["normal", "missing", "delay", "spike", "mixed"]); ax[2].set_title("Alert accuracy by fault scenario")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[7일 고강도 모의] 부대 8 x 품목 6 x 7일 = 336건, 시작 보유량 3일분(가상)
             시나리오  갱신(s)  경보  실제 부족   정확도  부족 탐지율
            정상 보고  0.041  23     29 0.964   0.690
        보고 누락 10%  0.039  23     29 0.964   0.690
        보고 지연 20%  0.037  22     29 0.961   0.655
         긴급 급증 5%  0.038  67     73 0.964   0.877
복합(누락10+지연20+급증5)  0.041  66     73 0.961   0.863
경보 정확도 최저 96.1% (목표 95% 이상: 달성)
부하 시험(부대 80개, 3360건) 갱신 0.07s, 최대 갱신 0.07s (목표 2초 이내: 달성)
품목별 첫 부족 발생(모의 n일차): ammo_C 4일차, ammo_D 5일차, fuel_jet 4일차
7일 탄약 총소모(가상 상자): 실제 4645, 예측 4633, 오차 -0.2%
경보 안전계수 상충관계(정상 보고): x1.0: 정확도 96.4%/탐지율 69%, x1.1: 정확도 94.3%/탐지율 90%, x1.2: 정확도 87.2%/탐지율 97% -> 부족 놓침을 줄이려면 안전계수를 올리고 정확도 저하를 감수한다
경보는 재보급 검토 요청이며 불출·우선순위 결정은 군수참모 결재를 거친다.
```
</details>
